# Experiment 04 — Circuit Depth Variations (QNG Optimized)

**Research question:** How does increasing the number of variational layers affect VQC classification performance when using the Quantum Natural Gradient (QNG)?

**Setup:** Fixed 2 features (cols 26, 4), 5000 samples, angle encoding.  
We vary the number of model layers: **1 → 2 → 3 → 4**.

### Parameters
| Layers | Params (2 qubits) |
|--------|-------------------|
| 1 | 1×2×3 + 1 = 7 |
| 2 | 2×2×3 + 1 = 13 |
| 3 | 3×2×3 + 1 = 19 |
| 4 | 4×2×3 + 1 = 25 |

In [8]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import pennylane as qml
from pennylane import numpy as pnp
from sklearn.metrics import roc_auc_score, roc_curve

from utils.data_utils import load_higgs, binary_accuracy

np.random.seed(42)

## 1. Setup

In [9]:
# Hyperparameters
N_FEATURES = 2
N_EPOCHS   = 30
BATCH_SIZE = 32
LR         = 0.05
N_SAMPLES  = 5000

LAYER_COUNTS = [1, 2, 3, 4]

X_train, X_val, X_test, y_train, y_val, y_test = load_higgs(
    path='../data/HIGGS.csv.gz',
    n_samples=N_SAMPLES,
    n_features=N_FEATURES,
    feature_indices=[26, 4],
    scale_range=(0, np.pi),
)

Selected features (cols [26, 4]): ['m_bb', 'missing energy mag.']
Dataset: 5000 samples | 2 features | train=3000, val=1000, test=1000


## 2. Training Engine (QNG)

In [10]:
def train_vqc_qng(n_features, n_layers, n_epochs=30, batch_size=32, lr=0.05):
    dev = qml.device('default.qubit', wires=n_features)

    @qml.qnode(dev, interface='autograd')
    def circuit(weights, x):
        for i in range(n_features):
            qml.RY(x[i], wires=i)
        for l in range(n_layers):
            for q in range(n_features):
                qml.Rot(weights[l, q, 0], weights[l, q, 1], weights[l, q, 2], wires=q)
            qml.CNOT(wires=[0, 1])
            qml.CNOT(wires=[1, 0])
        return qml.expval(qml.PauliZ(0))

    pnp.random.seed(42)
    w = pnp.array(np.random.uniform(0, 2*np.pi, (n_layers, n_features, 3)), requires_grad=True)
    b = pnp.array(0.0, requires_grad=True)

    opt_w = qml.QNGOptimizer(stepsize=lr)
    opt_b = qml.AdamOptimizer(stepsize=lr)
    mt_fn = qml.metric_tensor(circuit, approx='block-diag')

    val_losses = []

    for epoch in range(n_epochs):
        perm = np.random.permutation(len(X_train))
        X_shuf, y_shuf = X_train[perm], y_train[perm]

        for start in range(0, len(X_train), batch_size):
            Xb = X_shuf[start:start+batch_size]
            yb = y_shuf[start:start+batch_size].astype(float)

            def cost_w(weights):
                preds = pnp.array([circuit(weights, x) for x in Xb])
                return pnp.mean((yb - (preds + b)) ** 2)
            
            mt = mt_fn(w, Xb[0])
            w = opt_w.step(cost_w, w, metric_tensor_fn=lambda w: mt)

            def cost_b(bias):
                preds = pnp.array([circuit(w, x) for x in Xb])
                return pnp.mean((yb - (preds + bias)) ** 2)
            b = opt_b.step(cost_b, b)

        val_preds = np.array([float(circuit(w, x) + b) for x in X_val])
        vl_loss = np.mean((y_val.astype(float) - val_preds)**2)
        val_losses.append(vl_loss)

        if (epoch+1) % 10 == 0 or epoch == 0:
            print(f'  [Layers: {n_layers}] Epoch {epoch+1}/{n_epochs} | val_loss={vl_loss:.4f}')

    return circuit, w, b, val_losses

## 3. Run Experiments

In [11]:
results = {}

for nl in LAYER_COUNTS:
    print(f'\n{"="*50}')
    print(f'Running Depth: {nl} layers')
    
    circuit_fn, w_fin, b_fin, v_hist = train_vqc_qng(N_FEATURES, nl, N_EPOCHS, BATCH_SIZE, LR)
    
    # Evaluation
    test_raw = np.array([float(circuit_fn(w_fin, x) + b_fin) for x in X_test])
    acc = binary_accuracy(y_test, test_raw)
    y_test_01 = (y_test == 1).astype(int)
    test_scores = (test_raw - test_raw.min()) / (test_raw.max() - test_raw.min() + 1e-8)
    auc = roc_auc_score(y_test_01, test_scores)
    
    results[nl] = {
        'test_acc': acc,
        'test_auc': auc,
        'test_scores': test_scores,
        'val_losses': v_hist,
        'n_params': nl * N_FEATURES * 3 + 1
    }
    print(f'  → Final Result: Acc={acc:.4f}, AUC={auc:.4f}')


Running Depth: 1 layers
  [Layers: 1] Epoch 1/30 | val_loss=1.0008
  [Layers: 1] Epoch 10/30 | val_loss=1.1165
  [Layers: 1] Epoch 20/30 | val_loss=1.0222
  [Layers: 1] Epoch 30/30 | val_loss=1.0513
  → Final Result: Acc=0.5590, AUC=0.5310

Running Depth: 2 layers
  [Layers: 2] Epoch 1/30 | val_loss=0.9752
  [Layers: 2] Epoch 10/30 | val_loss=0.9710
  [Layers: 2] Epoch 20/30 | val_loss=0.9772
  [Layers: 2] Epoch 30/30 | val_loss=0.9699
  → Final Result: Acc=0.5990, AUC=0.6245

Running Depth: 3 layers
  [Layers: 3] Epoch 1/30 | val_loss=0.9627
  [Layers: 3] Epoch 10/30 | val_loss=0.9625
  [Layers: 3] Epoch 20/30 | val_loss=0.9788
  [Layers: 3] Epoch 30/30 | val_loss=0.9715
  → Final Result: Acc=0.6070, AUC=0.6236

Running Depth: 4 layers
  [Layers: 4] Epoch 1/30 | val_loss=0.9777
  [Layers: 4] Epoch 10/30 | val_loss=0.9547
  [Layers: 4] Epoch 20/30 | val_loss=0.9764
  [Layers: 4] Epoch 30/30 | val_loss=0.9803
  → Final Result: Acc=0.5880, AUC=0.6322


## 4. Results Summary

In [12]:
print('\nCircuit Depth Comparison (QNG Optimized)')
print(f'{"Layers":>8} {"Params":>8} {"Test Acc":>10} {"Test AUC":>10}')
print('-' * 42)
for nl, r in results.items():
    print(f'{nl:>8} {r["n_params"]:>8} {r["test_acc"]:>10.4f} {r["test_auc"]:>10.4f}')


Circuit Depth Comparison (QNG Optimized)
  Layers   Params   Test Acc   Test AUC
------------------------------------------
       1        7     0.5590     0.5310
       2       13     0.5990     0.6245
       3       19     0.6070     0.6236
       4       25     0.5880     0.6322
